# News Sentiment Analysis — DoubleML PLR
**Ashley Thompson — Capstone**

Replicates ml_analysis with `news_sent` as treatment. If the sign pattern matches
Twitter results, volume is not the driver — content and volume have opposing effects
regardless of channel.

- **Run 1:** News sentiment as continuous treatment
- **Run 2:** Negative news volume as continuous treatment

> **Runtime → Change runtime type → T4 GPU** before running.

## 1. Install dependencies

In [1]:
!pip install -q doubleml xgboost scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 603.5/603.5 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 68.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.5/224.5 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.2/55.2 kB 2.4 MB/s eta 0:00:00


## 2. Load data
Run Option A or Option B, then skip the other.

In [2]:
# Option A — Google Drive (data only)
from google.colab import drive
drive.mount('/content/drive')
DATA_PATH = '/content/drive/MyDrive/Colab Notebooks/Capstone/panel_long.csv'

Mounted at /content/drive


In [3]:
# GitHub output setup — run after whichever data option above
from google.colab import userdata
import os, subprocess

GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
REPO_DIR     = '/content/Capstone'
OUTPUT_PATH  = REPO_DIR + '/output/'

if os.path.exists(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
else:
    subprocess.run(
        ['git', 'clone',
         f'https://{GITHUB_TOKEN}@github.com/ATHOMP-03/Capstone.git',
         REPO_DIR],
        check=True,
    )
os.makedirs(OUTPUT_PATH, exist_ok=True)
print(f'Output path ready: {OUTPUT_PATH}')

Output path ready: /content/Capstone/output/


## 3. Setup

In [4]:
import os
import numpy as np
import pandas as pd
import doubleml as dml
from xgboost import XGBRegressor

np.random.seed(42)
os.makedirs(OUTPUT_PATH, exist_ok=True)

long = pd.read_csv(DATA_PATH, parse_dates=['date'])
print(f'Loaded {len(long):,} rows x {long.shape[1]} cols')
long.head()

Loaded 160,487 rows x 22 cols


,date,ticker,px_open,px_close,px_high,px_low,mkt_cap,total_equity,debt_to_equity,volume,...,news_sent,rsi_30,ma_50,twitter_neg_count,return,lag1,lag2,lag3,lag5,lag7
0,2025-01-01,A UN Equity,134.81,134.34,135.70,134.06,38366.8729,5898.0,60.5968,288566.0,...,-0.2921,45.9116,135.2216,0.0,-0.47,NaN,NaN,NaN,NaN,NaN
1,2025-01-02,A UN Equity,135.21,133.43,135.36,132.87,38106.9811,5898.0,60.5968,315308.0,...,-0.2921,44.9912,135.1550,0.0,-1.78,-0.47,NaN,NaN,NaN,NaN
2,2025-01-03,A UN Equity,133.45,135.69,136.04,132.86,38752.4265,5898.0,60.5968,399054.0,...,-0.3471,47.6856,135.1996,0.0,2.24,-1.78,-0.47,NaN,NaN,NaN
3,2025-01-06,A UN Equity,135.60,136.43,138.24,135.60,38963.7671,5898.0,60.5968,353095.0,...,-0.3471,48.5393,135.2676,0.0,0.83,2.24,-1.78,-0.47,NaN,NaN
4,2025-01-07,A UN Equity,136.83,137.41,140.28,136.83,39243.6504,5898.0,60.5968,359113.0,...,-0.2385,49.6647,135.4020,0.0,0.58,0.83,2.24,-1.78,NaN,NaN


In [5]:
def make_xgb():
    return XGBRegressor(
        n_estimators     = 500,
        learning_rate    = 0.05,
        max_depth        = 6,
        subsample        = 0.8,
        colsample_bytree = 0.8,
        device           = 'cuda',
        tree_method      = 'hist',
        random_state     = 42,
        n_jobs           = -1,
    )

## 4. Run 1 — News sentiment as continuous treatment
**Treatment:** `news_sent` | **Outcome:** `return`  
`twitter_sent` moves into the confounder set.

In [6]:
confounders_1 = [
    'px_high', 'px_low', 'mkt_cap', 'total_equity', 'debt_to_equity',
    'volume', 'twitter_sent', 'twitter_count', 'rsi_30', 'ma_50',
    'twitter_neg_count', 'lag1', 'lag2', 'lag3', 'lag5', 'lag7',
]

df1 = (
    long[['return', 'news_sent'] + confounders_1]
    .dropna()
    .reset_index(drop=True)
)

dml_1 = dml.DoubleMLPLR(
    dml.DoubleMLData(df1, y_col='return', d_cols='news_sent', x_cols=confounders_1),
    ml_l    = make_xgb(),
    ml_m    = make_xgb(),
    n_folds = 5,
    n_rep   = 20,
)
dml_1.fit()
dml_1.bootstrap(method='normal', n_rep_boot=1000)

print(f'N (complete cases): {len(df1):,}')
print(dml_1.summary)
print(dml_1.confint(level=0.95))

/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [15:09:48] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


N (complete cases): 144,509
               coef   std err        t         P>|t|     2.5 %    97.5 %
news_sent -0.759391  0.134525 -5.64498  1.652000e-08 -1.006927 -0.495727
              2.5 %    97.5 %
news_sent -1.006927 -0.495727


## 5. Run 2 — Negative news volume as continuous treatment
**Treatment:** `neg_news_vol = twitter_count × I(news_sent < 0)` | **Outcome:** `return`  
`news_sent` moves to confounders (determines treatment eligibility).

In [7]:
long['neg_news_vol'] = long['twitter_count'] * (long['news_sent'] < 0).astype(int)

confounders_2 = [
    'px_high', 'px_low', 'mkt_cap', 'total_equity', 'debt_to_equity',
    'volume', 'twitter_sent', 'news_sent', 'rsi_30', 'ma_50',
    'twitter_neg_count', 'lag1', 'lag2', 'lag3', 'lag5', 'lag7',
]

df2 = (
    long[['return', 'neg_news_vol'] + confounders_2]
    .dropna()
    .reset_index(drop=True)
)

dml_2 = dml.DoubleMLPLR(
    dml.DoubleMLData(df2, y_col='return', d_cols='neg_news_vol', x_cols=confounders_2),
    ml_l    = make_xgb(),
    ml_m    = make_xgb(),
    n_folds = 5,
    n_rep   = 20,
)
dml_2.fit()
dml_2.bootstrap(method='normal', n_rep_boot=1000)

print(f'N (complete cases): {len(df2):,}')
print(dml_2.summary)
print(dml_2.confint(level=0.95))

N (complete cases): 144,509
                  coef   std err         t     P>|t|     2.5 %    97.5 %
neg_news_vol  0.000588  0.000232  2.532496  0.011325  0.000157  0.001042
                 2.5 %    97.5 %
neg_news_vol  0.000157  0.001042


## 6. Export — Publication-ready LaTeX table
Saves `dml_news_results.tex` to your output folder with both runs side by side.

In [8]:
def dml_to_latex(models, labels, title, notes=''):
    """Build a publication-ready LaTeX table from DoubleMLPLR models."""
    def stars(p):
        if p < 0.01: return '***'
        if p < 0.05: return '**'
        if p < 0.1:  return '*'
        return ''

    all_treatments = []
    for m in models:
        for t in m.summary.index:
            if t not in all_treatments:
                all_treatments.append(t)

    ncols = len(models)
    col_fmt = 'l' + 'c' * ncols

    lines = [
        r'\begin{table}[htbp]',
        r'\centering',
        f'\\caption{{{title}}}',
        f'\\begin{{tabular}}{{{col_fmt}}}',
        r'\hline\hline',
        ' & ' + ' & '.join([f'({i+1}) {labels[i]}' for i in range(ncols)]) + r' \\',
        r'\hline',
    ]

    for t in all_treatments:
        t_label = t.replace('_', r'\_')
        coef_cells, se_cells = [], []
        for m in models:
            if t in m.summary.index:
                coef = m.summary.loc[t, 'coef']
                se   = m.summary.loc[t, 'std err']
                pval = m.summary.loc[t, 'P>|t|']
                coef_cells.append(f'{coef:.6f}{stars(pval)}')
                se_cells.append(f'({se:.6f})')
            else:
                coef_cells.append('---')
                se_cells.append('')
        lines.append(t_label + ' & ' + ' & '.join(coef_cells) + r' \\')
        lines.append('& ' + ' & '.join(se_cells) + r' \\')

    lines += [
        r'\hline',
        'Estimator & ' + ' & '.join(['DoubleML PLR'] * ncols) + r' \\',
        'ML Learner & ' + ' & '.join(['XGBoost (GPU)'] * ncols) + r' \\',
        r'\hline\hline',
    ]
    if notes:
        lines.append(
            f'\\multicolumn{{{ncols + 1}}}{{l}}'
            f'{{\\footnotesize \\textit{{Notes:}} {notes}}} \\\\'
        )
    lines += [r'\end{tabular}', r'\end{table}']
    return '\n'.join(lines)

In [9]:
tex = dml_to_latex(
    [dml_1, dml_2],
    ['News Sentiment', 'Neg News Volume'],
    title='DoubleML PLR --- News Sentiment and Stock Returns',
    notes=(
        'Heteroskedasticity-robust SE in parentheses. '
        'XGBoost learner with 5-fold cross-fitting, 20 repetitions, 1000 bootstrap draws. '
        'Run 2 treatment: neg\\_news\\_vol = twitter\\_count $\\times$ \\mathbb{1}(news\\_sent $<$ 0). '
        '*** p$<$0.01, ** p$<$0.05, * p$<$0.1.'
    ),
)
with open(OUTPUT_PATH + 'dml_news_results.tex', 'w') as f:
    f.write(tex)
print('Saved dml_news_results.tex')
print(tex)

Saved dml_news_results.tex
\begin{table}[htbp]
\centering
\caption{DoubleML PLR --- News Sentiment and Stock Returns}
\begin{tabular}{lcc}
\hline\hline
 & (1) News Sentiment & (2) Neg News Volume \\
\hline
news\_sent & -0.759391*** & --- \\
& (0.134525) &  \\
neg\_news\_vol & --- & 0.000588** \\
&  & (0.000232) \\
\hline
Estimator & DoubleML PLR & DoubleML PLR \\
ML Learner & XGBoost (GPU) & XGBoost (GPU) \\
\hline\hline
\multicolumn{3}{l}{\footnotesize \textit{Notes:} Heteroskedasticity-robust SE in parentheses. XGBoost learner with 5-fold cross-fitting, 20 repetitions, 1000 bootstrap draws. Run 2 treatment: neg\_news\_vol = twitter\_count $\times$ \mathbb{1}(news\_sent $<$ 0). *** p$<$0.01, ** p$<$0.05, * p$<$0.1.} \\
\end{tabular}
\end{table}


In [10]:
# Push outputs to GitHub
import os, subprocess

os.chdir(REPO_DIR)
subprocess.run(['git', 'config', 'user.email', 'ATHOMP-03@users.noreply.github.com'])
subprocess.run(['git', 'config', 'user.name',  'ATHOMP-03'])
subprocess.run(['git', 'add', 'output/'])
result = subprocess.run(['git', 'diff', '--cached', '--quiet'])
if result.returncode != 0:
    subprocess.run(['git', 'commit', '-m', 'Add news sentiment results from Colab'], check=True)
    subprocess.run(['git', 'push'], check=True)
    print('Pushed to GitHub.')
else:
    print('Nothing new to commit.')

Pushed to GitHub.
